# ActionShap — canonical reproducibility notebook

This notebook is a thin, auditable wrapper around the tracked command-line pipeline. It replaces the legacy pilot notebook whose synthetic gate and target-margin-as-NDCG labels were not valid final-paper evidence.

The final protocol:

1. runs an **independent 1,000-permutation convergence study** for every dataset/model pair and an NDCG-utility stress test;
2. uses primary ItemKNN and target-margin attribution while evaluating actions against a separate exact NDCG oracle;
3. runs the real-data masking gate before every attribution experiment;
4. evaluates five fixed seeds on two datasets and two history-conditioned models;
5. uses complete pre-test histories to construct unseen candidate sets;
6. permits abstention and actions of every size up to `B=2`;
7. generates hierarchical, distinct-user statistics and provenance manifests.

Raw result files are never interpreted directly in the manuscript. `scripts/make_paper_assets.py` rejects legacy schemas and incomplete experiment matrices.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import yaml

CODE_ROOT = Path.cwd().resolve()
if CODE_ROOT.name != "code" and (CODE_ROOT / "paper-ideas/ActionShap/code").is_dir():
    CODE_ROOT = (CODE_ROOT / "paper-ideas/ActionShap/code").resolve()
if not (CODE_ROOT / "scripts/run_final_suite.py").exists():
    raise RuntimeError("Open from the repository root or paper-ideas/ActionShap/code")
CONFIG_PATH = CODE_ROOT / "configs/final.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text())
print("CODE_ROOT:", CODE_ROOT)
print(yaml.safe_dump(CONFIG, sort_keys=False))


## Preflight

Both datasets must be supplied locally. MovieLens should be extracted to `code/data/ml-1m/ratings.dat`. Build Amazon Digital Music with `scripts/prepare_amazon_digital_music.py` at the path declared in `configs/final.yaml`. The suite deliberately refuses to claim final validation when either dataset or either model is absent.


In [ ]:
missing = []
for dataset in CONFIG["datasets"]:
    path = CODE_ROOT / dataset["path"]
    print(dataset["name"], "->", path, "OK" if path.exists() else "MISSING")
    if not path.exists():
        missing.append(path)
if missing:
    print("Supply the missing datasets before setting RUN_FINAL_SUITE=True.")


## Execute the complete suite

This is intentionally opt-in because a compliant run is computationally expensive. The runner performs convergence first and automatically raises the primary Monte Carlo budget when the predeclared thresholds require it.


In [ ]:
RUN_FINAL_SUITE = False
if RUN_FINAL_SUITE:
    subprocess.run(
        [sys.executable, str(CODE_ROOT / "scripts/run_final_suite.py"),
         "--config", str(CONFIG_PATH)],
        cwd=CODE_ROOT,
        check=True,
    )
else:
    subprocess.run(
        [sys.executable, str(CODE_ROOT / "scripts/run_final_suite.py"),
         "--config", str(CONFIG_PATH), "--dry-run"],
        cwd=CODE_ROOT,
        check=True,
    )


## Inspect validation

A paper result is usable only when this report says `PASS`. `PASS_WITH_WARNINGS` and `FAIL` must not be converted into manuscript claims.


In [ ]:
report_path = CODE_ROOT.parent / "paper/final/manifests/validation_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    if report["status"] != "PASS":
        print("STOP: final-paper acceptance criteria are not complete.")
else:
    print("No final validation report yet. Run the complete suite first.")


## Reproducibility notes

- Candidate, user-sampling, and tie-break seeds are fixed independently of the model seed.
- The manifest stores repository-relative source paths and SHA-256 hashes.
- Prefix-walk efficiency is reported only as a numerical identity.
- NDCG and target-margin effects are stored and labelled separately.
- Statistical inference averages repeated seeds within each user before bootstrap or permutation testing.
- Legacy assets under `paper/legacy_pilot/` are historical and cannot support final claims.
